# Advanced KPIs · Python/SQL validation

**Optional read-only step.** First complete SQL 12 and 13. Input: advanced_analysis.pkl and the advanced SQL summary view. All 12 comparisons should PASS, including NULL-aware rates and source-supplement counts.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [ ]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_context("advanced_analysis"))

## Reconcile advanced SQL summaries

Compare all 474 player-metric summaries, their coverage, rates, status and missing values against Python.

In [ ]:
import pyodbc
import numpy as np
import pandas as pd

# Use the same server/database configuration as the other SQL notebooks.
from scouting.database import connection_string as build_connection_string
connection = pyodbc.connect(build_connection_string(), timeout=10)

try:
    cursor = connection.cursor()
    cursor.execute("""
        SELECT *
        FROM analytics.vw_midfield_advanced_kpi_summary
        WHERE iteration_id = 743;
    """)

    sql_summary = pd.DataFrame.from_records(
        cursor.fetchall(),
        columns=[column[0] for column in cursor.description],
    )
finally:
    connection.close()

# Match each player and metric between Python and SQL.
python_summary = advanced_player_summary.rename(
    columns={"player.id": "player_id"}
).copy()

comparison = python_summary.merge(
    sql_summary,
    on=["player_id", "metric"],
    how="outer",
    suffixes=("_python", "_sql"),
    indicator=True,
    validate="one_to_one",
)

checks = [{
    "check": "Same 474 player-metric keys",
    "result": (
        "PASS"
        if len(comparison) == 474
        and comparison["_merge"].eq("both").all()
        else "FAIL"
    ),
}]

# Compare counts and status exactly.
for column in [
    "total_records",
    "populated_records",
    "unresolved_records",
    "supplemented_records",
    "data_status",
]:
    matches = comparison[f"{column}_python"].eq(
        comparison[f"{column}_sql"]
    )

    checks.append({
        "check": column,
        "result": "PASS" if matches.all() else "FAIL",
    })

# Allow tiny differences caused by stored numeric precision.
for column in [
    "available_value_sum",
    "midfield_minutes",
    "minutes_with_value",
    "minutes_with_value_pct",
    "observed_period_per90",
    "season_per90",
]:
    python_values = pd.to_numeric(
        comparison[f"{column}_python"], errors="raise"
    ).to_numpy(dtype=float, na_value=np.nan)

    sql_values = pd.to_numeric(
        comparison[f"{column}_sql"], errors="raise"
    ).to_numpy(dtype=float, na_value=np.nan)

    matches = np.isclose(
        python_values,
        sql_values,
        rtol=1e-9,
        atol=1e-5,
        equal_nan=True,
    )

    checks.append({
        "check": column,
        "result": "PASS" if matches.all() else "FAIL",
    })

validation = pd.DataFrame(checks)
display(validation)

if validation["result"].eq("PASS").all():
    print("SQL and Python advanced KPI summaries match.")
else:
    print("Differences found. Review the FAIL checks before continuing.")
if not validation["result"].eq("PASS").all():
    raise AssertionError("Advanced summaries differ; inspect FAIL checks.")


## Export advanced validation evidence

Save named intermediate tables for the next notebook. Generated files stay local and are excluded from Git.

In [ ]:
validation.to_csv(REPORTS_DIR / "advanced_python_sql_validation.csv", index=False)
